In [16]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import scipy
import matplotlib as mpl
from scipy.special import gamma

STARTING_RANGE_PARAMETER = 0.0001 # In [fm^-2]
ENDING_RANGE_PARAMETER = 2500
REDUCED_MASS = 935 * (4 / 5) # In [Mev / c^2], need to update value and units (10/11 A in MeV)
SUM_LIMIT = 25 # Determines the number of gaussians we expand our wave function to

V_LS = 11.71 # In MeV
DIFFUSIVITY = 0.6 # Diffusivity, may want to check the vaidity of this paticular number
r_0 = 1.2 # In fm, may want to chose a better value for small nuclei
A_C = 4 # The number of nucleons in the core

NUM_SUSY_GAUSSIANS = 15
SUSY_POTENTIAL_PARAMS = [0.015625,0.0216263,0.0299326,0.0414292,0.0573414,0.0793653,0.109848,
                                   0.152039,0.210435,0.291259,0.403127,0.557961,0.772264,1.06888,1.47942]
SUSY_POTENTIAL_COEFFS = np.array([-33.7795,125.728,-214.962,195.943,-59.5447,-118.117,235.946,-253.234,305.881,-243.858,148.467,
                         -74.1628,29.1132,-7.90282,1.0961])
CENTRAL_POTENTIAL_STRENGTH = -47.32 * 2

CENTRAL_POTENTIAL_PARAMETERS = [1 / ((2.30 * 1)**2)]

CENTRAL_MIXING_COEFFICIENTS = [1]

SIGMA = 2 #fm
TWO_PARTICLE_POTENTIAL_DEPTH = -24.22 # MeV


In [17]:
def anti_symm_normalisation(range_parameter_i, range_parameter_j):
    return 1 / np.sqrt(2 * (1 - single_particle_overlap(range_parameter_i, range_parameter_j)))

def single_particle_overlap(range_parameter_i, range_parameter_j, orbital_angular_momentum=0):
    term_1 = 2 * range_parameter_i * range_parameter_j
    term_2 = range_parameter_i**2 + range_parameter_j**2
    return (term_1 / term_2)**(2.5 + orbital_angular_momentum)

def single_particle_potential_element(range_parameter_i, range_parameter_j,
                                      potential_param, orbital_angular_momentum=0):
    """
    term_1 = 2 * range_parameter_i * range_parameter_j
    term_2 = range_parameter_i**2 + range_parameter_j**2 
    term_3 = range_parameter_i**2 * range_parameter_j**2 * potential_param
    return mixing_coefficient * potential_strength * (term_1 / (term_2 + term_3))**((5/2) + orbital_angular_momentum)
    """
    term_1 = (2 / (range_parameter_i * range_parameter_j))**(2.5 + orbital_angular_momentum)
    term_2 = (range_parameter_i**(-2) + range_parameter_j**(-2) + potential_param)**(2.5 + orbital_angular_momentum)
    return  term_1 / term_2

def single_particle_kinetic_element(range_parameter_i, range_parameter_j, orbital_angular_momentum=0, μ=REDUCED_MASS):
    term_A = (2**((7/2) + orbital_angular_momentum)) / (3 + 2 * orbital_angular_momentum)

    term_1 = ((orbital_angular_momentum + 0) * (orbital_angular_momentum + 1)) / (range_parameter_i * range_parameter_j)
    term_2 = ((range_parameter_i * range_parameter_j) / (range_parameter_i**2 + range_parameter_j**2))**(1.5 + orbital_angular_momentum)
    term_B = term_1 * term_2

    term_3 = (range_parameter_i * range_parameter_j)**(0.5 + orbital_angular_momentum)
    term_4 = (range_parameter_i**2 + range_parameter_j**2)**((7/2) + orbital_angular_momentum)
    term_5 = (orbital_angular_momentum + 1) * (orbital_angular_momentum + 2) * (range_parameter_i**4 + range_parameter_j**4)
    term_6 = (11 + 2 * orbital_angular_momentum * (5 + orbital_angular_momentum)) * range_parameter_i**2 * range_parameter_j**2
    term_C = (term_3 / term_4) * (term_5 - term_6)

    return (197**2 / (2 * μ)) * (term_A * (term_B - term_C))

def single_particle_r_minus_two_element(range_parameter_i, range_parameter_j, orbital_angular_momentum=0, μ=REDUCED_MASS):
    term_1 = (2**((9/2) + orbital_angular_momentum)) / ((3 + 2 * orbital_angular_momentum) * range_parameter_i * range_parameter_j)
    term_2 = ((range_parameter_i * range_parameter_j) / (range_parameter_i**2 + range_parameter_j**2))**((3/2) + orbital_angular_momentum)
    return (197**2 / (2 * μ)) * term_1 * term_2

def overlap_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l):
    term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_overlap(range_parameter_j, range_parameter_l)
    term_2 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_overlap(range_parameter_j, range_parameter_k)
    term_3 = anti_symm_normalisation(range_parameter_i, range_parameter_j) * anti_symm_normalisation(range_parameter_k, range_parameter_l)
    mat_elem = 2 * (term_1 - term_2) / term_3

    return mat_elem

def kinetic_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l, μ=REDUCED_MASS, orb_ang_momentum=0):
    term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_kinetic_element(
        range_parameter_j, range_parameter_l)
    term_2 = single_particle_overlap(range_parameter_j, range_parameter_l) * single_particle_kinetic_element(
        range_parameter_i, range_parameter_k)
    term_3 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_kinetic_element(
        range_parameter_j, range_parameter_k)
    term_4 = single_particle_overlap(range_parameter_j, range_parameter_k) * single_particle_kinetic_element(
        range_parameter_i, range_parameter_l)
    term_5 = anti_symm_normalisation(range_parameter_i, range_parameter_j) * anti_symm_normalisation(
        range_parameter_k, range_parameter_l)

    mat_elem = (term_1 + term_2 - term_3 - term_4) / term_5
    return mat_elem

def potential_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l, central_potential_mixing_coefficient,
                             central_potential_param, V_0=CENTRAL_POTENTIAL_STRENGTH):
    term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_potential_element(
        range_parameter_j, range_parameter_l, central_potential_param)
    term_2 = single_particle_overlap(range_parameter_j, range_parameter_l) * single_particle_potential_element(
        range_parameter_i, range_parameter_k, central_potential_param)
    term_3 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_potential_element(
        range_parameter_j, range_parameter_k, central_potential_param)
    term_4 = single_particle_overlap(range_parameter_j, range_parameter_k) * single_particle_potential_element(
        range_parameter_i, range_parameter_l, central_potential_param)
    term_5 = anti_symm_normalisation(range_parameter_i, range_parameter_j) * anti_symm_normalisation(
        range_parameter_k, range_parameter_l)

    mat_elem = (term_1 + term_2 - term_3 - term_4) / term_5
    return V_0 * central_potential_mixing_coefficient * mat_elem

def r_two_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l):
    term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_r_minus_two_element(
        range_parameter_j, range_parameter_l)
    term_2 = single_particle_overlap(range_parameter_j, range_parameter_l) * single_particle_r_minus_two_element(
        range_parameter_i, range_parameter_k)
    term_3 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_r_minus_two_element(
        range_parameter_j, range_parameter_k)
    term_4 = single_particle_overlap(range_parameter_j, range_parameter_k) * single_particle_r_minus_two_element(
        range_parameter_i, range_parameter_l)
    term_5 = anti_symm_normalisation(range_parameter_i, range_parameter_j) * anti_symm_normalisation(
        range_parameter_k, range_parameter_l)

    mat_elem = (term_1 + term_2 - term_3 - term_4) / term_5
    return mat_elem

def distinguishable_interaction_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l,
                                               potential_parameter, potential_strength=TWO_PARTICLE_POTENTIAL_DEPTH):
    def single_particle_norm(range_parameter):
        return (128 / (9 * np.pi**3 * range_parameter**10))**(0.25)

    term_A = range_parameter_i**(-2) + range_parameter_k**(-2) + potential_parameter**(-2)
    term_B = range_parameter_j**(-2) + range_parameter_l**(-2) + potential_parameter**(-2)
    term_1 = 2 + 3 * term_A * term_B * potential_parameter**4
    term_2 = (term_A * term_B * potential_parameter**4 - 1)**(3.5)
    term_3 = single_particle_norm(range_parameter_i) * single_particle_norm(
        range_parameter_j) * single_particle_norm(range_parameter_k) * single_particle_norm(
        range_parameter_l)
    return -(np.pi**3 * 3 * potential_parameter**10 * term_1 * term_3) / (4 * term_2)
    
    np.pi**3 * potential_strength * single_particle_norm(range_parameter_i) * single_particle_norm(
        range_parameter_j) * single_particle_norm(range_parameter_k) * single_particle_norm(
        range_parameter_l) * term_1

def interaction_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l, potential_parameter,
                              potential_strength=TWO_PARTICLE_POTENTIAL_DEPTH):
    term_1 = distinguishable_interaction_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k,
                                                          range_parameter_l, potential_parameter)
    term_2 = distinguishable_interaction_matrix_element(range_parameter_j, range_parameter_i, range_parameter_l,
                                                         range_parameter_k, potential_parameter)
    term_3 = distinguishable_interaction_matrix_element(range_parameter_i, range_parameter_j, range_parameter_l,
                                                          range_parameter_k, potential_parameter)
    term_4 = distinguishable_interaction_matrix_element(range_parameter_j, range_parameter_i, range_parameter_k,
                                                          range_parameter_l, potential_parameter)
    mat_elem = anti_symm_normalisation(range_parameter_i, range_parameter_j) * anti_symm_normalisation(
        range_parameter_k, range_parameter_l) * (term_1 + term_2 - term_3 - term_4)
    return mat_elem

In [18]:
def two_particle_basis_matrix_generation(central_mixing_coefficients=CENTRAL_MIXING_COEFFICIENTS,
                                         central_potential_parameters=CENTRAL_POTENTIAL_PARAMETERS,
                                         susy_potential_mixing_coefficients=SUSY_POTENTIAL_COEFFS,
                                         susy_potential_parameters=SUSY_POTENTIAL_PARAMS,
                                         size=SUM_LIMIT, potential_parameter=SIGMA):
    
    mat_size = int(size * (size - 1) * 0.5)

    first_h_matrix = np.zeros(shape=(mat_size, mat_size))
    second_h_matrix = np.zeros(shape=(mat_size, mat_size))
    interaction_matrix = np.zeros(shape=(mat_size, mat_size))
    overlap_matrix = np.zeros(shape=(mat_size, mat_size))

    A = -1
    B = -1

    for I in range(1, size**2 + 1):
        i = np.mod((I - 1), size) + 1
        i_range_parameter = next_range_parameter(i - 1)
        j = np.floor((I - 1) / size) + 1
        j_range_parameter = next_range_parameter(j - 1)
        if i < j:
            A += 1
            B = -1
    
            for J in range(1, size**2 + 1):
                k = np.mod((J - 1), size) + 1
                k_range_parameter = next_range_parameter(k - 1)
                l = np.floor((J - 1) / size) + 1
                l_range_parameter = next_range_parameter(l - 1)
                if k < l:
                    B += 1

                    overlap_matrix[A, B] = overlap_matrix_element(i_range_parameter, j_range_parameter,
                                                                        k_range_parameter, l_range_parameter)

                    interaction_matrix[A, B] = interaction_matrix_element(i_range_parameter, j_range_parameter,
                                                                          k_range_parameter, l_range_parameter,
                                                                          potential_parameter)

                    first_potential_energy_term = 0
                    first_susy_potential_term = 0
                    first_kinetic_term = kinetic_matrix_element(i_range_parameter, j_range_parameter,
                                                                k_range_parameter, l_range_parameter)
                    first_r_square_term = r_two_matrix_element(i_range_parameter, j_range_parameter,
                                                               k_range_parameter, l_range_parameter)

                    for m in range(len(central_mixing_coefficients)):
                        first_potential_energy_term += potential_matrix_element(
                            i_range_parameter, j_range_parameter, k_range_parameter, l_range_parameter,
                            central_mixing_coefficients[m], central_potential_parameters[m])
                    for m in range(len(susy_potential_mixing_coefficients)):
                        first_susy_potential_term += potential_matrix_element(
                            i_range_parameter, j_range_parameter, k_range_parameter, l_range_parameter,
                            susy_potential_mixing_coefficients[m], susy_potential_parameters[m],
                            V_0=1)
                    first_h_matrix[A, B] = (first_kinetic_term + first_potential_energy_term + first_susy_potential_term + first_r_square_term)

                    second_potential_energy_term = 0
                    second_susy_potential_term = 0
                    second_kinetic_term = kinetic_matrix_element(i_range_parameter, j_range_parameter,
                                                                 k_range_parameter, l_range_parameter)
                    second_r_square_term = r_two_matrix_element(i_range_parameter, j_range_parameter,
                                                               k_range_parameter, l_range_parameter)                    
                    for m in range(len(central_mixing_coefficients)):
                        second_potential_energy_term += potential_matrix_element(
                            i_range_parameter, j_range_parameter, k_range_parameter, l_range_parameter,
                            central_mixing_coefficients[m], central_potential_parameters[m])
                    for m in range(len(susy_potential_mixing_coefficients)):
                        second_susy_potential_term += potential_matrix_element(
                            i_range_parameter, j_range_parameter, k_range_parameter, l_range_parameter,
                            susy_potential_mixing_coefficients[m], susy_potential_parameters[m],
                            V_0=1)
                    second_h_matrix[A, B] = (second_kinetic_term + second_potential_energy_term + second_susy_potential_term + second_r_square_term)
                else:
                    continue

        else:
            continue
        if A % 10 == 0:
            print(A)
                

    np.savetxt('nmatrix.csv', overlap_matrix)
    np.savetxt('h1matrix.csv', first_h_matrix)
    np.savetxt('h2matrix.csv', second_h_matrix)
    np.savetxt('interaction.csv', interaction_matrix)
    
    return first_h_matrix + second_h_matrix + interaction_matrix, overlap_matrix


def interaction_mat_gen(size=SUM_LIMIT, potential_parameter=SIGMA):
    interaction_matrix = np.zeros(shape=(size**2, size**2))

    for I in range(1, size**2 + 1):
        i = np.mod((I - 1), size) + 1
        i_range_parameter = next_range_parameter(i - 1)
        j = np.floor((I - 1) / size) + 1
        j_range_parameter = next_range_parameter(j - 1)
        for J in range(1, size**2 + 1):
            k = np.mod((J - 1), size) + 1
            k_range_parameter = next_range_parameter(k - 1)
            l = np.floor((J - 1) / size) + 1
            l_range_parameter = next_range_parameter(l - 1)

            interaction_matrix[I - 1, J - 1] = distinguishable_interaction_matrix_element(i_range_parameter, j_range_parameter,
                                                    k_range_parameter, l_range_parameter, potential_parameter)
    np.savetxt('interaction11.csv', interaction_matrix)
    return interaction_matrix


def next_range_parameter(i, starting_range_parameter=STARTING_RANGE_PARAMETER, ending_range_parameter=ENDING_RANGE_PARAMETER,
                         sum_limit=SUM_LIMIT):
    """
    Finds the next range parameter given the previous and initial range parameters.
    Currently using a simple geometric series to determine range parameters.
    Chose geometric basis parameters $\alpha_i = \alpha_1a^{i-1}$ with initial parameters $\alpha_1 = 0.01, a=2$

    Parameters
    ----------
    i : int detailing the iteration number

    Returns
    -------
    new_range_parameter: float

    """
    geometric_progression_number = (ending_range_parameter / starting_range_parameter)**(1 / (sum_limit - 1))
    new_range_parameter = starting_range_parameter * geometric_progression_number**(i)

    return new_range_parameter
h_matrix, n_matrix = two_particle_basis_matrix_generation()
int_mat = interaction_mat_gen()

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290


In [19]:
s_eigenvalues, s_eigenvectors = scipy.linalg.eigh(h_matrix, n_matrix)
#bound_states = np.real(s_eigenvalues) < 0
s_overlap_eigenvalues, s_overlap_eigenvectors = scipy.linalg.eigh(n_matrix)
s_overlap_matrix_condition_number = np.max(s_overlap_eigenvalues) / np.min(s_overlap_eigenvalues)
print(f"The s 1/2 overlap matrix condition number is", s_overlap_matrix_condition_number)

s0_eigenvector = np.asmatrix(s_eigenvectors[:, 0])
s1_eigenvector = np.asmatrix(s_eigenvectors[:, 1])
print("The S state eigenvalues are", s_eigenvalues)
#print(f'The bound states have energies {bound_states}')
#print("The S1 eigenvector is", s1_eigenvector)
print(f'The bound states have energies{np.real(np.sort(s_eigenvalues[np.real(s_eigenvalues) < 0]))}')

The s 1/2 overlap matrix condition number is 362.30729682919593
The S state eigenvalues are [-2.80331888e-01 -2.80250914e-01 -2.79920568e-01 -2.78558708e-01
 -2.72926296e-01 -2.49545133e-01 -1.51148752e-01  1.20526257e-04
  4.50887795e-04  5.31837597e-04  1.81272743e-03  1.89370944e-03
  2.22401308e-03  7.44514992e-03  7.52613915e-03  7.85642292e-03
  9.21832001e-03  3.08265005e-02  3.09072996e-02  3.12374596e-02
  3.25995027e-02  3.82319128e-02  1.29223438e-01  1.29304356e-01
  1.29634721e-01  1.30996587e-01  1.36629000e-01  1.60010099e-01
  2.39897520e-01  5.20291502e-01  5.20372478e-01  5.20702821e-01
  5.22064682e-01  5.27697092e-01  5.51078240e-01  6.49471607e-01
  2.46321619e+00  2.74463205e+00  2.74471301e+00  2.74504332e+00
  2.74640520e+00  2.75203757e+00  2.77541705e+00  2.87378979e+00
  3.26447839e+00  1.67535391e+01  1.70532343e+01  1.70533153e+01
  1.70536456e+01  1.70550074e+01  1.70606389e+01  1.70839930e+01
  1.71818440e+01  1.75692575e+01  1.97724139e+01  1.12849764e+0